In [0]:
from LDCDataAccessLayerPy import KeyVaultManager, SharePointManager, SqlManager, databricks_init
from datetime import datetime, timedelta
from LDCDataAccessLayerPy import databricks_init, DataLakeManagerGen2
from io import BytesIO
import LDCDataAccessLayerPy
#Initiate the secret to access KeyVault secrets
databricks_init(dbutils, 'GO')
sp = SharePointManager()
sql_mgr = SqlManager()

import logging
logger = spark._jvm.org.apache.log4j
logging.getLogger("py4j").setLevel(logging.ERROR)

import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import ListedColormap
from sklearn.cluster import KMeans
from openpyxl import load_workbook

from datetime import datetime, timedelta
import re
from dateutil.relativedelta import relativedelta
from LDCDataAccessLayerPy import PriceManager, graph
from LDCDataAccessLayerPy import ZemaManager
import math
import plotly.express as px

zema = ZemaManager()


# USD ARS

In [0]:
import pandas as pd
import requests
from io import BytesIO

url = "https://www.bcra.gob.ar/Pdfs/PublicacionesEstadisticas/com3500.xls"

# Get the content of the file
response = requests.get(url, verify=False)
response.raise_for_status()  # Raise an error if download failed

# Load it into pandas
df = pd.read_excel(BytesIO(response.content))


daily_df = df.iloc[2:, 2:]
daily_df.columns = daily_df.iloc[0]     # Set first row as header
daily_df = daily_df[1:].reset_index(drop=True)  # Drop first row and reset index
daily_df.rename(
    columns={
        daily_df.columns[0]: 'date',
        daily_df.columns[1]: 'FX'
    },
    inplace=True
)
daily_df=daily_df[['date','FX']]

daily_df['date'] = pd.to_datetime(daily_df['date'])
daily_df_index=daily_df.set_index('date')
weekly_df = daily_df_index.resample('W').mean().reset_index()
monthly_df = daily_df_index.resample('M').mean().reset_index()
yearly_df = daily_df_index.resample('Y').mean().reset_index()

sp.save_pd_to_excel('sites/grainsargprojects/models/Sorghum%20Miling/FX ARSUSD.xlsx',daily_df)


# PIZARRA

In [0]:
query = "select * from [GO].[AR_Consiagro_Prices_v1];"
sql = SqlManager()
db = "ldc-rep-ds"
price_df = sql.sql_query(db, query)
display(price_df)


In [0]:
sp.save_pd_to_excel("sites/grainsargprojects/models/Sorghum%20Miling/Wheat pizarra.xlsx",price_df,index=False)

In [0]:
wheat=price_df[['Date','Wheat','Wheat_Currency','Region']]
sorghum=price_df[['Date','Sorghum','Sorghum_Currency','Region']]
soybean=price_df[['Date','Soybean','Soybean_Currency','Region']]
sunflower=price_df[['Date','Sunflower','Sunflower_Currency','Region']]

wheat=wheat[wheat.Region=='Buenos Aires']
wheat_ars=wheat[wheat.Wheat_Currency=='ARS']
wheat_usd=wheat[wheat.Wheat_Currency=='USD']

sorghum=sorghum[sorghum.Region=='Rosario']
sorghum_ars=sorghum[(sorghum.Sorghum_Currency=='ARS')|(sorghum.Sorghum_Currency.isna())]
sorghum_usd=sorghum[sorghum.Sorghum_Currency=='USD']

soybean=soybean[soybean.Region=='Buenos Aires']
soybean_ars=soybean[soybean.Soybean_Currency=='ARS']
soybean_usd=soybean[soybean.Soybean_Currency=='USD']

sunflower=sunflower[sunflower.Region=='Buenos Aires']
sunflower_ars=sunflower[sunflower.Sunflower_Currency=='ARS']
sunflower_usd=sunflower[sunflower.Sunflower_Currency=='USD']




In [0]:
grain_cols = ['Wheat']

# Convert columns to numeric
for col in grain_cols:
    wheat_ars[col] = pd.to_numeric(wheat_ars[col], errors='coerce')

# Convert date to datetime and sort
wheat_ars['Date'] = pd.to_datetime(wheat_ars['Date'])
wheat_ars = wheat_ars.sort_values('Date').reset_index(drop=True)
# Step-by-step fill: loop row by row
for i in range(len(wheat_ars)):
    for col in grain_cols:
        val = wheat_ars.at[i, col]

        if pd.isna(val) or val == 0:
            # Get up to 5 previous values (excluding current row)
            prev_values = wheat_ars.loc[max(i - 5, 0):i - 1, col]

            # Remove NaNs and zeros from the average
            prev_values = prev_values[prev_values != 0].dropna()

            # Fill with average or 0 if no valid previous data
            if not prev_values.empty:
                wheat_ars.at[i, col] = prev_values.mean()
            else:
                wheat_ars.at[i, col] = 0

grain_cols = ['Sorghum']
for col in grain_cols:
    sorghum_ars[col] = pd.to_numeric(sorghum_ars[col], errors='coerce')
sorghum_ars['Date'] = pd.to_datetime(sorghum_ars['Date'])
sorghum_ars = sorghum_ars.sort_values('Date').reset_index(drop=True)

for i in range(len(sorghum_ars)):
    for col in grain_cols:
        val = sorghum_ars.at[i, col]

        if pd.isna(val) or val == 0:
            prev_values = sorghum_ars.loc[max(i - 5, 0):i - 1, col]
            prev_values = prev_values[prev_values != 0].dropna()
            if not prev_values.empty:
                sorghum_ars.at[i, col] = prev_values.mean()
            else:
                sorghum_ars.at[i, col] = 0



grain_cols = ['Soybean']
for col in grain_cols:
    soybean_ars[col] = pd.to_numeric(soybean_ars[col], errors='coerce')
soybean_ars['Date'] = pd.to_datetime(soybean_ars['Date'])
soybean_ars = soybean_ars.sort_values('Date').reset_index(drop=True)

for i in range(len(soybean_ars)):
    for col in grain_cols:
        val = soybean_ars.at[i, col]

        if pd.isna(val) or val == 0:
            prev_values = soybean_ars.loc[max(i - 5, 0):i - 1, col]
            prev_values = prev_values[prev_values != 0].dropna()
            if not prev_values.empty:
                soybean_ars.at[i, col] = prev_values.mean()
            else:
                soybean_ars.at[i, col] = 0


grain_cols = ['Sunflower']
for col in grain_cols:
    sunflower_ars[col] = pd.to_numeric(sunflower_ars[col], errors='coerce')
sunflower_ars['Date'] = pd.to_datetime(sunflower_ars['Date'])
sunflower_ars = sunflower_ars.sort_values('Date').reset_index(drop=True)

for i in range(len(sunflower_ars)):
    for col in grain_cols:
        val = sunflower_ars.at[i, col]

        if pd.isna(val) or val == 0:
            prev_values = sunflower_ars.loc[max(i - 5, 0):i - 1, col]
            prev_values = prev_values[prev_values != 0].dropna()
            if not prev_values.empty:
                sunflower_ars.at[i, col] = prev_values.mean()
            else:
                sunflower_ars.at[i, col] = 0



# SORGO TO USD

In [0]:
sorghum_ars.rename(columns={'Date': "date"}, inplace=True)

In [0]:
sorghum_ars_merged=pd.merge(sorghum_ars,daily_df,on='date')
sorghum_ars_merged['sorg_usd']=sorghum_ars_merged['Sorghum']/sorghum_ars_merged['FX']
# Sort by the 'Date' column (ascending)
sorghum_ars_merged = sorghum_ars_merged.sort_values(by='date')


In [0]:
monthly_avg = sorghum_ars_merged.set_index('date')['sorg_usd'].resample('M').mean()

# Convert the index to the first day of the month
monthly_avg.index = monthly_avg.index.to_period('M').to_timestamp()

# Reset index and rename columns for clarity
monthly_avg = monthly_avg.reset_index()
monthly_avg.columns = ['date', 'Avg_sorg_usd']
monthly_avg
 

# MILING DATA

In [0]:

milling=sp.read_pd_from_excel("sites/grainsargprojects/models/Sorghum%20Miling/Commercial Stocks.xlsx",sheet_name='Molienda',header=2)
milling_sorgo=milling[['MY Sorgo','MY','TOTAL SORGO']]
milling_sorgo.rename(columns={'MY': "date"}, inplace=True)
milling_sorgo

In [0]:
mill_w_price=pd.merge(milling_sorgo,monthly_avg,on='date')



In [0]:
correlation_not_filtered = mill_w_price['Avg_sorg_usd'].corr(mill_w_price['TOTAL SORGO'])
print(f"Correlation between price and volume: {correlation_not_filtered:.2f}")

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Scatter plot
plt.figure(figsize=(8, 5))
sns.scatterplot(data=mill_w_price, x='Avg_sorg_usd', y='TOTAL SORGO')
plt.title('Sorghum Milling vs Price')
plt.xlabel('Average Sorghum Price (USD)')
plt.ylabel('Total Sorghum Milling')
plt.grid(True)
plt.show()


In [0]:
import pandas as pd

# Compute IQR for each variable
Q1_price = mill_w_price['Avg_sorg_usd'].quantile(0.25)
Q3_price = mill_w_price['Avg_sorg_usd'].quantile(0.75)
IQR_price = Q3_price - Q1_price

Q1_volume = mill_w_price['TOTAL SORGO'].quantile(0.25)
Q3_volume = mill_w_price['TOTAL SORGO'].quantile(0.75)
IQR_volume = Q3_volume - Q1_volume

# Filter out outliers
mill_w_price_filtered = mill_w_price[
    (mill_w_price['Avg_sorg_usd'] >= Q1_price - 1.5 * IQR_price) & (mill_w_price['Avg_sorg_usd'] <= Q3_price + 1.5 * IQR_price) &
    (mill_w_price['TOTAL SORGO'] >= Q1_volume - 1.5 * IQR_volume) & (mill_w_price['TOTAL SORGO'] <= Q3_volume + 1.5 * IQR_volume)
]

# Pearson correlation
correlation = mill_w_price_filtered['Avg_sorg_usd'].corr(mill_w_price_filtered['TOTAL SORGO'])
print(f"Correlation between price and volume: {correlation:.2f}")


In [0]:
# Scatter plot
plt.figure(figsize=(8, 5))
sns.scatterplot(data=mill_w_price_filtered, x='Avg_sorg_usd', y='TOTAL SORGO')
plt.title('Sorghum Milling vs Price')
plt.xlabel('Average Sorghum Price (USD)')
plt.ylabel('Total Sorghum Milling')
plt.grid(True)
plt.show()

In [0]:
import statsmodels.api as sm

X = mill_w_price_filtered['Avg_sorg_usd']
y = mill_w_price_filtered['TOTAL SORGO']

# Add constant for intercept
X = sm.add_constant(X)

# Fit regression model
model = sm.OLS(y, X).fit()

# Print summary
print(model.summary())
